# AthleteIQ Feature Engineering

## Objective

Prepare the processed dataset for machine learning by separating the target variable from the predictor variables, encoding categorical features, and creating training and testing datasets.

### Tasks

- Load the processed dataset
- Inspect the processed data
- Define the prediction target
- Separate features and target
- Identify numerical and categorical features
- Encode categorical variables
- Split the data into training and testing sets
- Verify the final feature matrix

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv("../data/processed/processed_sleep_health_dataset.csv")

df.head()

,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic BP,Diastolic BP
0,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,NaN,126,83
1,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
3,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90
4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90


In [3]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (374, 13)

Columns:
['Gender', 'Age', 'Occupation', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Heart Rate', 'Daily Steps', 'Sleep Disorder', 'Systolic BP', 'Diastolic BP']


In [4]:
print(df.isnull().sum())

Gender                       0
Age                          0
Occupation                   0
Sleep Duration               0
Quality of Sleep             0
Physical Activity Level      0
Stress Level                 0
BMI Category                 0
Heart Rate                   0
Daily Steps                  0
Sleep Disorder             219
Systolic BP                  0
Diastolic BP                 0
dtype: int64


In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 242


## Prediction Target

The prediction target is `Quality of Sleep`.

The model will attempt to predict a participant's reported sleep quality using lifestyle, behavioral, and physiological characteristics.

`Quality of Sleep` is therefore separated from the predictor variables.

In [6]:
# Define the target variable
y = df["Quality of Sleep"]

# Define the predictor variables
X = df.drop(columns=["Quality of Sleep"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (374, 12)
y shape: (374,)


In [7]:
print(y.value_counts().sort_index())

Quality of Sleep
4      5
5      7
6    105
7     77
8    109
9     71
Name: count, dtype: int64


In [8]:
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']

Numerical features:
['Age', 'Sleep Duration', 'Physical Activity Level', 'Stress Level', 'Heart Rate', 'Daily Steps', 'Systolic BP', 'Diastolic BP']


## Feature Types

### Numerical Features

- Age
- Sleep Duration
- Physical Activity Level
- Stress Level
- Heart Rate
- Daily Steps
- Systolic BP
- Diastolic BP

### Categorical Features

- Gender
- Occupation
- BMI Category
- Sleep Disorder

The categorical features need to be converted into numerical representations before they can be used by most machine learning algorithms.

In [9]:
# Create the OneHotEncoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit and transform the categorical features
X_categorical = encoder.fit_transform(X[categorical_features])

# Convert encoded features to a DataFrame
encoded_columns = encoder.get_feature_names_out(categorical_features)

X_categorical = pd.DataFrame(
    X_categorical,
    columns=encoded_columns,
    index=X.index
)

X_categorical.head()

,Gender_Female,Gender_Male,Occupation_Accountant,Occupation_Doctor,Occupation_Engineer,Occupation_Lawyer,Occupation_Manager,Occupation_Nurse,Occupation_Sales Representative,Occupation_Salesperson,Occupation_Scientist,Occupation_Software Engineer,Occupation_Teacher,BMI Category_Normal,BMI Category_Obese,BMI Category_Overweight,Sleep Disorder_Insomnia,Sleep Disorder_Sleep Apnea,Sleep Disorder_nan
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [10]:
X_numerical = X[numerical_features].copy()

X_processed = pd.concat(
    [X_numerical, X_categorical],
    axis=1
)

print("Processed feature matrix shape:", X_processed.shape)
X_processed.head()

Processed feature matrix shape: (374, 27)


,Age,Sleep Duration,Physical Activity Level,Stress Level,Heart Rate,Daily Steps,Systolic BP,Diastolic BP,Gender_Female,Gender_Male,...,Occupation_Salesperson,Occupation_Scientist,Occupation_Software Engineer,Occupation_Teacher,BMI Category_Normal,BMI Category_Obese,BMI Category_Overweight,Sleep Disorder_Insomnia,Sleep Disorder_Sleep Apnea,Sleep Disorder_nan
0,27,6.1,42,6,77,4200,126,83,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,28,6.2,60,8,75,10000,125,80,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,28,6.2,60,8,75,10000,125,80,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,28,5.9,30,8,85,3000,140,90,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,28,5.9,30,8,85,3000,140,90,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [11]:
print(X_processed.dtypes.value_counts())

float64    20
int64       7
Name: count, dtype: int64


In [12]:
print("Non-numeric columns:")
print(
    X_processed.select_dtypes(
        exclude=["int64", "float64"]
    ).columns.tolist()
)

Non-numeric columns:
[]


In [13]:
print("Missing values in X_processed:")
print(X_processed.isnull().sum().sum())

print("\nMissing values in y:")
print(y.isnull().sum())

Missing values in X_processed:
0

Missing values in y:
0


## Train-Test Split

The dataset will be divided into:

- 80% training data
- 20% testing data

The training set will be used to train the machine learning models, while the testing set will be kept separate for final evaluation.

A fixed `random_state` is used so that the split is reproducible.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (299, 27)
X_test: (75, 27)
y_train: (299,)
y_test: (75,)


In [15]:
print("Training target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

Training target distribution:
Quality of Sleep
4     3
5     6
6    79
7    61
8    91
9    59
Name: count, dtype: int64

Testing target distribution:
Quality of Sleep
4     2
5     1
6    26
7    16
8    18
9    12
Name: count, dtype: int64


In [16]:
# Save processed feature matrices
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

# Save target variables
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Feature-engineered datasets saved successfully.")

Feature-engineered datasets saved successfully.


In [17]:
X_train_check = pd.read_csv("../data/processed/X_train.csv")
X_test_check = pd.read_csv("../data/processed/X_test.csv")
y_train_check = pd.read_csv("../data/processed/y_train.csv")
y_test_check = pd.read_csv("../data/processed/y_test.csv")

print("X_train:", X_train_check.shape)
print("X_test:", X_test_check.shape)
print("y_train:", y_train_check.shape)
print("y_test:", y_test_check.shape)

X_train: (299, 27)
X_test: (75, 27)
y_train: (299, 1)
y_test: (75, 1)


In [18]:
print("===== FINAL DAY 5 CHECK =====")

print("Original dataset:", df.shape)
print("Processed feature matrix:", X_processed.shape)

print("\nTraining data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nMissing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

print("\nNon-numeric feature columns:")
print(
    X_processed.select_dtypes(
        exclude=["int64", "float64"]
    ).columns.tolist()
)

print("\n===== FEATURE ENGINEERING COMPLETE =====")

===== FINAL DAY 5 CHECK =====
Original dataset: (374, 13)
Processed feature matrix: (374, 27)

Training data:
X_train: (299, 27)
y_train: (299,)

Testing data:
X_test: (75, 27)
y_test: (75,)

Missing values in X_train: 0
Missing values in X_test: 0

Non-numeric feature columns:
[]

===== FEATURE ENGINEERING COMPLETE =====


## Day 5 Summary

The dataset was transformed into a machine-learning-ready format.

### Completed

- Loaded the processed dataset.
- Defined `Quality of Sleep` as the prediction target.
- Separated predictor variables (`X`) from the target (`y`).
- Identified numerical and categorical features.
- Applied one-hot encoding to categorical variables.
- Combined numerical and encoded categorical features.
- Verified that all model features are numeric.
- Checked for missing values.
- Split the dataset into 80% training and 20% testing sets.
- Saved the training and testing datasets for reproducibility.

### Result

The project now has separate training and testing datasets that can be used to build and evaluate machine learning models.

The next stage is to establish a baseline model and compare machine learning algorithms.